# Phase II: Preprocessing
**Minor Project: Predictive Classification using Random Forest Ensembles**

Continues from the EDA notebook. Steps: (1) handle missing values, (2) encode categorical data, (3) stratified 80:20 train/test split, (4) feature-scaling decision. Output: `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` for Phase III.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid')
os.makedirs('prep_figures', exist_ok=True)
RANDOM_STATE = 42
CSV = 'forest_cover.csv'
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
else:
    from sklearn.datasets import fetch_covtype
    df = fetch_covtype(as_frame=True).frame
    df.to_csv(CSV, index=False)
print('Raw shape:', df.shape)

## Step 1: Missing / null values

In [ ]:
nulls = df.isnull().sum()
print('Total missing cells:', int(nulls.sum()))
print(nulls[nulls > 0] if nulls.sum() else 'No column has missing values.')

NUMERIC = ['Elevation','Aspect','Slope','Horizontal_Distance_To_Hydrology','Vertical_Distance_To_Hydrology',
           'Horizontal_Distance_To_Roadways','Hillshade_9am','Hillshade_Noon','Hillshade_3pm',
           'Horizontal_Distance_To_Fire_Points']
# Safeguard: median imputation (robust to the skewed distance features). It only acts if nulls exist.
if nulls.sum() > 0:
    df[NUMERIC] = df[NUMERIC].fillna(df[NUMERIC].median())
print('Missing after cleaning:', int(df.isnull().sum().sum()))

# Sanity checks on value ranges (Aspect 0-360, hillshade 0-255, slope 0-90)
checks = {'Aspect':(0,360),'Hillshade_9am':(0,255),'Hillshade_Noon':(0,255),'Hillshade_3pm':(0,255),'Slope':(0,90)}
for c,(lo,hi) in checks.items():
    bad = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f'{c:15s} out-of-range rows: {bad}')
print('Exact duplicate rows:', df.duplicated().sum(), '(kept: repeated measurements are valid)')

**Decision:** nulls are imputed with the median only if present. Outliers are kept (valid terrain extremes; trees are insensitive to them).

## Step 2: Target creation and encoding

In [ ]:
# Binary target: Cottonwood/Willow (Cover_Type 4) vs the rest
df['Target'] = (df['Cover_Type'] == 4).astype(int)

# Rebuild one readable categorical column from the 4 binary wilderness columns
wild_cols = [c for c in df.columns if c.startswith('Wilderness_Area')]
names = ['Rawah','Neota','Comanche_Peak','Cache_la_Poudre']
df['Wilderness_Area'] = pd.Series(df[wild_cols].values.argmax(axis=1)).map(dict(enumerate(names))).values
print(df['Wilderness_Area'].value_counts())

In [ ]:
# One-Hot encoding of the nominal categorical feature.
# Label encoding would impose a false order (Rawah < Neota < ...), which is wrong for nominal data.
wild_ohe = pd.get_dummies(df['Wilderness_Area'], prefix='Wilderness', dtype=int)
soil_cols = [c for c in df.columns if c.startswith('Soil_Type')]   # already one-hot (40 binary columns)

# IMPORTANT: Cover_Type is dropped from the features. Target is derived from it, so keeping it = data leakage.
X = pd.concat([df[NUMERIC], wild_ohe, df[soil_cols]], axis=1)
y = df['Target']
print('Feature matrix:', X.shape, '| columns by group:',
      {'numeric':len(NUMERIC), 'wilderness_ohe':wild_ohe.shape[1], 'soil':len(soil_cols)})
assert 'Cover_Type' not in X.columns and 'Target' not in X.columns
X.head()

## Step 3: Stratified 80:20 train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)   # stratify keeps the rare class in both sets

summary = pd.DataFrame({
    'rows':[len(X_train), len(X_test)],
    'positives':[int(y_train.sum()), int(y_test.sum())],
    '% positive':[y_train.mean()*100, y_test.mean()*100]}, index=['Train (80%)','Test (20%)']).round(3)
print(summary)

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['Overall','Train','Test'], [y.mean()*100, y_train.mean()*100, y_test.mean()*100], color=['#999','#4C72B0','#C44E52'])
ax.set_ylabel('% Cottonwood/Willow'); ax.set_title('Class share preserved by stratification')
plt.tight_layout(); plt.savefig('prep_figures/06_stratified_split.png', dpi=300); plt.show()

## Step 4: Feature scaling decision (evidence, not just theory)
Random Forest splits on thresholds, so monotonic rescaling cannot change which rows fall left/right of a split. The check below fits the same forest on raw and `StandardScaler`-scaled features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

n = min(50000, len(X_train))
Xs, ys = X_train.iloc[:n], y_train.iloc[:n]
scaler = StandardScaler().fit(Xs[NUMERIC])
Xs_sc = Xs.copy(); Xs_sc[NUMERIC] = scaler.transform(Xs[NUMERIC])
Xt_sc = X_test.copy(); Xt_sc[NUMERIC] = scaler.transform(X_test[NUMERIC])

res = {}
for name, (a, b) in {'Raw features':(Xs, X_test), 'Scaled features':(Xs_sc, Xt_sc)}.items():
    rf = RandomForestClassifier(n_estimators=50, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE).fit(a, ys)
    res[name] = f1_score(y_test, rf.predict(b))
print(pd.Series(res).round(4).rename('Test F1'))

**Decision:** scaling is **skipped**. The F1 scores above are (near-)identical, so scaling adds no benefit and would only make feature values harder to interpret (metres/degrees become z-scores).

## Save the preprocessed data for Phase III

In [ ]:
X_train.to_csv('X_train.csv', index=False); X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False);  y_test.to_csv('y_test.csv', index=False)
print('Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv')

## Phase II summary
| Step | Decision | Reason |
|---|---|---|
| Missing values | Median imputation as safeguard | Robust to skew; dataset normally has none |
| Outliers / duplicates | Kept | Valid terrain extremes and repeated measurements |
| Encoding | One-hot for wilderness area; soil already one-hot | Nominal data, no false ordering |
| Leakage control | `Cover_Type` removed from features | Target is derived from it |
| Split | Stratified 80:20, `random_state=42` | Rare class must appear in both sets |
| Scaling | Skipped (verified on test F1) | Trees are threshold-based, not distance-based |